# 14.04 · Image generation & diffusion

**Chapter:** Multimodal AI · **Milestone:** M7

## Goals
- [ ] Diffusion intuition: add noise step by step, learn to remove it — **build a tiny diffusion model from scratch** on 2-D data
- [ ] Classifier-free guidance (CFG): how a prompt steers generation, and what the guidance scale does
- [ ] Latent diffusion: VAE + denoiser (U-Net / DiT) + text encoder; few-step distilled models (Turbo, LCM)
- [ ] Generate locally with 🤗 `diffusers` (text-to-image, image-to-image) and evaluate prompt alignment with CLIP score
- [ ] Know the landscape: open models (FLUX, SD 3.5, Qwen-Image), APIs (Gemini/Imagen, GPT-Image, Midjourney), ComfyUI, LoRAs, licensing

## Resources
- [HF Diffusion Models course](https://github.com/huggingface/diffusion-models-class) · [diffusers docs](https://huggingface.co/docs/diffusers) — free
- [Lilian Weng – What are diffusion models?](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/) — free
- [DDPM paper](https://arxiv.org/abs/2006.11239) · [Classifier-free guidance](https://arxiv.org/abs/2207.12598) · [Latent diffusion](https://arxiv.org/abs/2112.10752) — free
- [ComfyUI](https://github.com/comfyanonymous/ComfyUI) — free/OSS

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_moons
from torch import nn

from aieng import DATA

torch.manual_seed(0)
OUT = DATA / "images"
OUT.mkdir(exist_ok=True)

## 1. Diffusion from scratch, in 2-D
Data: points on two interleaved half-moons, labelled 0 and 1 (our "prompt"). The **forward process** mixes data with Gaussian noise over
T steps until it's pure noise; it has a closed form: `x_t = √ᾱ_t · x_0 + √(1-ᾱ_t) · ε`. The model learns the **reverse**: given a noisy point,
the step `t` and the label, predict the noise `ε` that was added. Generation = start from noise and remove predicted noise step by step.

In [ ]:
X, y = make_moons(20_000, noise=0.05, random_state=0)
X = torch.tensor((X - X.mean(0)) / X.std(0), dtype=torch.float32)
y = torch.tensor(y)

T = 200
betas = torch.linspace(1e-4, 0.03, T)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t in zip(axes, [0, 20, 50, 100, 199]):
    xt = abar[t].sqrt() * X[:2000] + (1 - abar[t]).sqrt() * torch.randn(2000, 2)
    ax.scatter(*xt.T, s=1, c=y[:2000], cmap="coolwarm"); ax.set_title(f"t = {t}"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle("forward process: data → noise"); plt.show()

The denoiser is a small MLP that sees `(x_t, t, label)`. For **classifier-free guidance** we randomly drop the label 15% of the time
(replace it with a "null" label 2), so the same network learns both the conditional and the unconditional noise prediction.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.t_emb, self.y_emb = nn.Embedding(T, d), nn.Embedding(3, d)  # labels 0, 1 and 2 = "no label"
        self.net = nn.Sequential(nn.Linear(2 + d, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU(), nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 2))

    def forward(self, x, t, label):
        return self.net(torch.cat([x, self.t_emb(t) + self.y_emb(label)], dim=1))


model = Denoiser()
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
start, losses = time.perf_counter(), []
for step in range(6000):
    idx = torch.randint(len(X), (512,))
    x0, label = X[idx], y[idx].clone()
    label[torch.rand(512) < 0.15] = 2                       # label dropout -> learns unconditional too
    t = torch.randint(T, (512,))
    eps = torch.randn_like(x0)
    xt = abar[t, None].sqrt() * x0 + (1 - abar[t, None]).sqrt() * eps
    loss = (model(xt, t, label) - eps).pow(2).mean()         # the whole DDPM objective: predict the noise
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
print(f"trained in {time.perf_counter() - start:.0f}s, final loss {np.mean(losses[-200:]):.3f}")

**Sampling** (DDPM): start at `x_T ~ N(0, I)`; at each step subtract the predicted noise and add a little fresh noise.
**CFG**: `ε = ε_uncond + w · (ε_cond − ε_uncond)`. w = 0 ignores the label, w = 1 is plain conditional, w > 1 pushes harder toward the label
(sharper, less diverse — the same knob as `guidance_scale` in Stable Diffusion).

In [ ]:
@torch.no_grad()
def sample(n, label, w=1.0, keep=()):
    x, snaps = torch.randn(n, 2), {}
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        e_c = model(x, tt, torch.full((n,), label))
        e_u = model(x, tt, torch.full((n,), 2))
        eps = e_u + w * (e_c - e_u)
        x = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x += betas[t].sqrt() * torch.randn_like(x)
        if t in keep:
            snaps[t] = x.clone()
    return x, snaps


_, snaps = sample(2000, label=0, keep=(199, 150, 100, 50, 0))
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (t, s) in zip(axes, snaps.items()):
    ax.scatter(*s.T, s=1); ax.set_title(f"t = {t}"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle("reverse process (label 0): noise → data"); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, w in zip(axes, [0.0, 1.0, 3.0, 8.0]):
    for label, color in [(0, "tab:blue"), (1, "tab:red")]:
        s, _ = sample(1000, label, w)
        ax.scatter(*s.T, s=1, c=color)
    ax.set_title(f"guidance w = {w}"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.suptitle("classifier-free guidance: blue = 'prompt 0', red = 'prompt 1'"); plt.show()

w = 0: both "prompts" give the same mixture (the label is ignored). w = 1: each label lands on its own moon. Large w: tighter, less diverse,
eventually pushed off the data — the 2-D version of over-saturated, "fried" images at high guidance.

## 2. From 2-D points to images: latent diffusion
Same algorithm, three upgrades:
1. **Latent space**: a VAE compresses a 512×512×3 image into a 64×64×4 latent (48× fewer numbers); diffusion runs there, the VAE decodes at the end.
2. **Denoiser**: a U-Net (SD 1.x/2.x/XL) or a **Diffusion Transformer** (DiT: SD 3, FLUX, Qwen-Image) instead of an MLP.
3. **Conditioning**: a text encoder (CLIP, T5, or an LLM) turns the prompt into embeddings the denoiser attends to (cross-attention).

Newer models train with **flow matching** (a straighter path from noise to data than DDPM), and **distillation** (Turbo/ADD, LCM, Lightning)
cuts sampling from 25–50 steps to 1–4. We use **SD-Turbo** (SD 2.1 distilled, ~2.5 GB download, fast on a Mac; non-commercial-friendly
Stability license — check before shipping).

In [ ]:
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", dtype=dtype, variant="fp16").to(device)
pipe.set_progress_bar_config(disable=True)

prompt = "a watercolor painting of a llama in the Andes at sunrise"
images, times = [], []
for steps in (1, 2, 4):
    t = time.perf_counter()
    images.append(pipe(prompt, num_inference_steps=steps, guidance_scale=0.0, generator=torch.Generator().manual_seed(0)).images[0])
    times.append(time.perf_counter() - t)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
for ax, img, steps, secs in zip(axes, images, (1, 2, 4), times):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"{steps} step(s), {secs:.1f}s")
plt.show()

Turbo models are trained to work *without* CFG (`guidance_scale=0.0`), which also halves the compute (no unconditional pass).
Regular models use 20–50 steps and `guidance_scale` 3–7.

## 3. Same prompt, different seeds — and measuring prompt alignment
Generation is sampling: the seed picks the starting noise. To compare prompts or settings fairly, fix seeds and generate several images each.
**CLIP score** = cosine similarity between CLIP's embedding of the image and of the prompt: a cheap automatic check of "does the image show
what was asked?" (it can't judge quality or aesthetics; use human ratings or preference models for that).

In [ ]:
from sentence_transformers import SentenceTransformer

clip = SentenceTransformer("clip-ViT-B-32")


def clip_score(img, text):
    a, b = clip.encode([img], normalize_embeddings=True), clip.encode([text], normalize_embeddings=True)
    return float((a @ b.T)[0, 0])


prompts = ["a red bicycle leaning against a blue door", "a cat wearing sunglasses on a beach", "an isometric 3d render of a tiny coffee shop"]
fig, axes = plt.subplots(len(prompts), 4, figsize=(12, 9.5))
for row, p in enumerate(prompts):
    for col, seed in enumerate(range(4)):
        img = pipe(p, num_inference_steps=2, guidance_scale=0.0, generator=torch.Generator().manual_seed(seed)).images[0]
        img.save(OUT / f"p{row}-s{seed}.png")
        axes[row, col].imshow(img); axes[row, col].axis("off")
        axes[row, col].set_title(f"seed {seed} · CLIP {clip_score(img, p):.2f}", fontsize=9)
    axes[row, 0].text(-20, 256, p, rotation=90, va="center", ha="right", fontsize=8)
plt.tight_layout(); plt.show()
mismatch = pipe(prompts[0], num_inference_steps=2, guidance_scale=0.0, generator=torch.Generator().manual_seed(0)).images[0]
print(f"CLIP(bicycle image, bicycle prompt) = {clip_score(mismatch, prompts[0]):.2f} vs (bicycle image, cat prompt) = {clip_score(mismatch, prompts[1]):.2f}")

Look closely at the bicycle row: some seeds swap the colours (blue bicycle, red door) and CLIP scores them the same as the correct ones.
CLIP is a bag-of-concepts matcher: good at "is there a bicycle and a door", bad at **attribute binding**, counting and spatial relations.
For those, use a VLM judge (14.01: "Is the bicycle red and the door blue? yes/no") or benchmarks like GenEval / T2I-CompBench.

## 4. Image-to-image
Start from an existing image + noise instead of pure noise. `strength` = how much noise (0 = keep the input, 1 = ignore it). A crude drawing
becomes a picture; the same mechanism powers inpainting (noise only a masked region) and style transfer.

In [ ]:
from PIL import Image, ImageDraw

sketch = Image.new("RGB", (512, 512), "#87ceeb")
d = ImageDraw.Draw(sketch)
d.rectangle([0, 360, 512, 512], fill="#2e8b57"); d.ellipse([380, 40, 470, 130], fill="#ffd700")
d.polygon([(120, 360), (220, 200), (320, 360)], fill="#8b4513"); d.rectangle([180, 300, 260, 360], fill="#ffffff")

img2img = AutoPipelineForImage2Image.from_pipe(pipe)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(sketch); axes[0].set_title("input sketch")
for ax, strength in zip(axes[1:], (0.5, 0.75, 0.95)):
    out = img2img("a cozy wooden cabin in a green valley on a sunny day, photo", image=sketch, strength=strength,
                  num_inference_steps=4, guidance_scale=0.0, generator=torch.Generator().manual_seed(0)).images[0]
    ax.imshow(out); ax.set_title(f"strength {strength}")
for ax in axes: ax.axis("off")
plt.show()

Low strength mostly keeps flat colours (with 4 steps, strength 0.5 means only 2 denoising steps run); the sketch turns into a scene only when
most of it is re-noised. Steps actually run = `int(num_inference_steps × strength)`, so with 4 steps 0.75 and 0.95 both run 3 —
that's why they look alike; use more steps for finer control. Pick strength per task: ~0.3 for light retouching, 0.6–0.8 for "keep the layout, change the look".

## 5. The landscape
| | Examples | Notes |
|---|---|---|
| Open weights | FLUX.1 [schnell] (Apache-2.0) / [dev] (non-commercial), SD 3.5, SDXL, Qwen-Image, HiDream, Sana | `diffusers`; 8–24 GB VRAM for the big ones (quantized versions run on Macs) |
| APIs | Gemini image (Nano Banana) / Imagen, OpenAI GPT-Image, Midjourney, Ideogram, Recraft, Black Forest Labs API | best quality and text rendering, editing by instruction |
| Tools | ComfyUI (node graphs), Draw Things / DiffusionBee (Mac apps), InvokeAI | workflows: ControlNet, inpainting, upscaling |
| Customisation | **LoRA** (a style/subject in ~10 MB), DreamBooth, IP-Adapter (image prompt), ControlNet (pose/edges/depth) | train LoRAs with `diffusers` scripts, kohya, ai-toolkit |

Video (Veo, Sora, Kling, open: Wan, HunyuanVideo, LTX-Video) uses the same ideas with space-time latents.

**Responsibility**: respect licences (many open weights restrict commercial use), don't generate real people without consent, label AI images
(C2PA content credentials, SynthID-style watermarks), and keep a safety filter on public-facing generators.

## Exercises
1. Extend the 2-D model to three classes with `make_blobs`, or replace DDPM sampling with **DDIM** (deterministic, 20 steps instead of 200).
   Plot sample quality vs number of steps.
2. Generate images locally with diffusers; try a **LoRA** style: SD-Turbo is SD 2.1-based, so search the Hub for SD 2.1 LoRAs and load one with
   `pipe.load_lora_weights(...)`. Or switch to `stabilityai/sdxl-turbo` (larger) where many more LoRAs exist.
3. Build a tiny prompt eval: 10 prompts × 4 seeds, scored by CLIP **and** a VLM yes/no judge (14.01) per setting (steps 1/2/4, prompt with/without style words). Which change helps most?
4. Install ComfyUI (`uvx comfy-cli install`), rebuild the img2img workflow as a node graph and compare the experience with code.

In [ ]:
# your code here